# 05. SQL Analysis

## 1. 분석 목적

03번 EDA에서 pandas로 계산한 집단별 이직 현황을 **SQL로 다시 집계**해, 같은 결과가 나오는지 교차 검증한다.
또한 SQL로 다음 두 가지를 추가로 집계한다.

- 현 직무 근속연수(`YearsInCurrentRole`) 구간별 이직률
- 직무 × 초과근무 교차 이직률

이직률은 모두 **이직자 수(분자) ÷ 직원 수(분모) × 100**이며 소수점 둘째 자리로 표시한다.
집단 간 차이는 연관성이며 인과관계나 통계적 유의성을 뜻하지 않는다.

> **데이터 출처·라이선스:** IBM HR Analytics Employee Attrition & Performance(IBM 데이터 과학자가 만든 가상 데이터. [Kaggle](https://www.kaggle.com/datasets/pavansubhasht/ibm-hr-analytics-attrition-dataset), [IBM/employee-attrition-aif360](https://github.com/IBM/employee-attrition-aif360)). 데이터베이스는 [ODbL 1.0](https://opendatacommons.org/licenses/odbl/1-0/), 개별 내용은 [DbCL 1.0](https://opendatacommons.org/licenses/dbcl/1-0/)으로 표기되어 있습니다. 이 Notebook의 출력(표·그림·수치)은 이 데이터에서 만든 결과물이며, MIT License(`../LICENSE`)는 코드에만 적용됩니다. 전처리 CSV(ODbL 1.0)와 전처리 방법은 `../data/README.md`를 참고하세요.
>
> Contains information from the IBM HR Analytics Employee Attrition & Performance dataset, which is made available here under the Open Database License (ODbL) 1.0.

## 2. SQL 분석 환경

- **SQL 엔진:** SQLite 3. Python 표준 라이브러리 `sqlite3`를 사용하므로 추가 설치가 없다.
- **데이터베이스:** 메모리(`:memory:`)에 만들고 분석이 끝나면 닫는다. 파일로 저장하지 않는다.
- **쿼리:** 모두 [`05_sql_analysis.sql`](05_sql_analysis.sql)에 있다.
  - 각 쿼리는 `-- name: <이름>` 줄로 구분된다. 이 Notebook은 파일을 읽어 그 표시로 나눠 실행한다.
  - 쿼리는 Notebook 안에 다시 쓰지 않는다.
- **출력:** 집계값만 출력하고, 직원별 원본 행은 출력하지 않는다.
- **입력:** `../data/raw/hr.csv` 원본을 읽기만 한다.

In [1]:
import re
import sqlite3
from pathlib import Path

import pandas as pd

DATA_PATH = Path("../data/raw/hr.csv")
SQL_PATH = Path("05_sql_analysis.sql")

print("SQLite:", sqlite3.sqlite_version)
print("pandas:", pd.__version__)

SQLite: 3.45.3
pandas: 2.2.3


## 3. CSV 데이터 로딩

원본 CSV를 그대로 읽는다. 형태와 결측치 수만 확인하고 직원별 행은 출력하지 않는다.

In [2]:
hr = pd.read_csv(DATA_PATH, encoding="utf-8-sig")

print("행, 열:", hr.shape)
print("전체 결측치 수:", int(hr.isna().sum().sum()))

행, 열: (1470, 35)
전체 결측치 수: 0


## 4. SQLite 메모리 데이터베이스 생성
## 5. 데이터 테이블 생성

메모리 데이터베이스에 원본 그대로 `hr` 테이블을 만든다. 정수 열은 `INTEGER`, 문자열 열은 `TEXT`로 저장된다.

In [3]:
con = sqlite3.connect(":memory:")
hr.to_sql("hr", con, index=False)

print("hr 테이블 행 수:", con.execute("SELECT COUNT(*) FROM hr").fetchone()[0])

hr 테이블 행 수: 1470


## 6. SQL 파일 불러오기 및 실행

SQL 파일을 `-- name:` 표시로 나눠 모든 쿼리를 차례로 실행하고 결과를 `results`에 담는다.

In [4]:
sql_text = SQL_PATH.read_text(encoding="utf-8")
parts = re.split(r"^-- name: (\w+)\s*$", sql_text, flags=re.M)
queries = {parts[i]: parts[i + 1].strip() for i in range(1, len(parts), 2)}

results = {name: pd.read_sql_query(sql, con) for name, sql in queries.items()}

print("실행한 쿼리:", len(results))
for name, frame in results.items():
    print(f"  {name}: {len(frame)}행")

실행한 쿼리: 16
  q1a_row_count: 1행
  q1b_columns: 35행
  q1c_duplicate_employee_number: 1행
  q1d_missing_values: 9행
  q1e_attrition_values: 2행
  q2_overall: 1행
  q3_department: 3행
  q4_gender: 2행
  q5_job_role: 9행
  q6a_overtime: 2행
  q6b_overtime_comparison: 1행
  q7_age_group: 5행
  q8a_tenure_years_at_company: 5행
  q8b_years_in_current_role: 5행
  q8c_tenure_mean_by_attrition: 2행
  q9_job_role_overtime: 18행


## 7. 쿼리별 결과

### Q1. 데이터 기본 점검

In [5]:
display(results["q1a_row_count"])
display(results["q1c_duplicate_employee_number"])
display(results["q1d_missing_values"])
display(results["q1e_attrition_values"])

,row_count
0,1470


,row_count,distinct_employee_number,duplicate_count
0,1470,1470,0


,column_name,null_count,blank_count
0,EmployeeNumber,0,0
1,Age,0,0
2,Attrition,0,0
3,Department,0,0
4,Gender,0,0
5,JobRole,0,0
6,OverTime,0,0
7,YearsAtCompany,0,0
8,YearsInCurrentRole,0,0


,Attrition,employee_count
0,No,1233
1,Yes,237


컬럼 목록(`q1b_columns`)은 35개이므로 개수와 자료형 분포만 확인한다.

In [6]:
columns = results["q1b_columns"]
print("컬럼 수:", len(columns))
display(columns["sqlite_type"].value_counts().rename_axis("sqlite_type").reset_index(name="column_count"))

컬럼 수: 35


,sqlite_type,column_count
0,INTEGER,26
1,TEXT,9


- 행은 1,470개, 컬럼은 35개(정수 26개, 문자열 9개)다.
- `EmployeeNumber` 중복은 0건이다.
- 주요 컬럼 9개에 결측치와 빈 문자열이 없다.
- `Attrition`은 `No` 1,233건, `Yes` 237건이다.

### Q2. 전체 이직률

In [7]:
display(results["q2_overall"])

,employee_count,attrition_count,retained_count,attrition_rate
0,1470,237,1233,16.12


전체 1,470명 중 237명이 이직해 이직률은 16.12%다.

### Q3. 부서별 이직률

In [8]:
display(results["q3_department"])

,Department,employee_count,attrition_count,attrition_rate,overall_rate,diff_pp,small_sample
0,Sales,446,92,20.63,16.12,4.51,N
1,Human Resources,63,12,19.05,16.12,2.93,N
2,Research & Development,961,133,13.84,16.12,-2.28,N


| 부서 | 이직률 | 전체 평균(16.12%) 대비 | 비고 |
|---|---:|---:|---|
| Sales | 20.63% | +4.51%p | |
| Human Resources | 19.05% | +2.93%p | 직원 63명이라 비율 해석에 주의 |
| Research & Development | 13.84% | −2.28%p | |

### Q4. 성별 이직률

In [9]:
display(results["q4_gender"])

,Gender,employee_count,attrition_count,attrition_rate
0,Male,882,150,17.01
1,Female,588,87,14.80


남성 17.01%(150/882), 여성 14.80%(87/588)로 차이는 2.21%p다.

### Q5. 직무별 이직률

In [10]:
display(results["q5_job_role"])

,JobRole,employee_count,attrition_count,attrition_rate,small_sample
0,Sales Representative,83,33,39.76,N
1,Laboratory Technician,259,62,23.94,N
2,Human Resources,52,12,23.08,N
3,Sales Executive,326,57,17.48,N
4,Research Scientist,292,47,16.10,N
5,Manufacturing Director,145,10,6.90,N
6,Healthcare Representative,131,9,6.87,N
7,Manager,102,5,4.90,N
8,Research Director,80,2,2.50,N


- Sales Representative가 39.76%(33/83)로 가장 높고, Research Director가 2.50%(2/80)로 가장 낮다.
- 가장 작은 직무도 52명(Human Resources)이라 30명 미만 직무는 없다.

### Q6. 초과근무 분석

In [11]:
display(results["q6a_overtime"])
display(results["q6b_overtime_comparison"])

,OverTime,employee_count,attrition_count,attrition_rate
0,Yes,416,127,30.53
1,No,1054,110,10.44


,overtime_rate,no_overtime_rate,diff_pp,rate_ratio
0,30.53,10.44,20.09,2.93


- 초과근무 집단은 30.53%(127/416), 비초과근무 집단은 10.44%(110/1,054)다.
- 차이는 20.09%p이며, 초과근무 집단의 이직률이 약 2.93배다.

### Q7. 연령대별 이직률

In [12]:
display(results["q7_age_group"])

,AgeGroup,employee_count,attrition_count,attrition_rate,small_sample
0,Under 25,97,38,39.18,N
1,25-34,554,112,20.22,N
2,35-44,505,51,10.10,N
3,45-54,245,25,10.20,N
4,55+,69,11,15.94,N


- 25세 미만이 39.18%로 가장 높고, 35~44세가 10.10%로 가장 낮다.
- 55세 이상은 15.94%로, 연령이 높을수록 일정하게 낮아지지는 않는다.

### Q8. 근속기간별 이직률

In [13]:
display(results["q8a_tenure_years_at_company"])
display(results["q8b_years_in_current_role"])
display(results["q8c_tenure_mean_by_attrition"])

,TenureGroup,employee_count,attrition_count,attrition_rate,small_sample
0,0-1 Years,215,75,34.88,N
1,2-3 Years,255,47,18.43,N
2,4-5 Years,306,40,13.07,N
3,6-10 Years,448,55,12.28,N
4,Over 10 Years,246,20,8.13,N


,RoleTenureGroup,employee_count,attrition_count,attrition_rate,small_sample
0,0-1 Years,301,84,27.91,N
1,2-3 Years,507,84,16.57,N
2,4-5 Years,140,16,11.43,N
3,6-10 Years,444,48,10.81,N
4,Over 10 Years,78,5,6.41,N


,Attrition,employee_count,mean_years_at_company,mean_years_in_current_role
0,No,1233,7.37,4.48
1,Yes,237,5.13,2.90


| 기준 | 가장 높은 구간 | 가장 낮은 구간 |
|---|---|---|
| 회사 근속(`YearsAtCompany`) | 0~1년 34.88%(75/215) | 10년 초과 8.13%(20/246) |
| 현 직무 근속(`YearsInCurrentRole`) | 0~1년 27.91%(84/301) | 10년 초과 6.41%(5/78) |

- `YearsInCurrentRole` 구간은 기존 분석에 정의가 없어 근속 구간과 같은 경계를 적용했다. 기존 보고서에는 없는 새 집계다.
- 평균 근속은 재직 7.37년, 이직 5.13년이다. 현 직무 근속은 재직 4.48년, 이직 2.90년이다.

### Q9. 직무 × 초과근무 교차 분석

In [14]:
display(results["q9_job_role_overtime"])

,JobRole,OverTime,employee_count,attrition_count,attrition_rate,small_sample
0,Healthcare Representative,No,94,7,7.45,N
1,Healthcare Representative,Yes,37,2,5.41,N
2,Human Resources,No,39,7,17.95,N
3,Human Resources,Yes,13,5,38.46,Y
4,Laboratory Technician,No,197,31,15.74,N
5,Laboratory Technician,Yes,62,31,50.00,N
6,Manager,No,75,1,1.33,N
7,Manager,Yes,27,4,14.81,Y
8,Manufacturing Director,No,106,6,5.66,N
9,Manufacturing Director,Yes,39,4,10.26,N


- **같은 직무 안에서의 차이:** 대부분의 직무에서 초과근무 집단의 이직률이 더 높다.
  - Laboratory Technician: 50.00%(31/62) 대 15.74%
  - Research Scientist: 34.02% 대 7.18%
  - Sales Executive: 32.98% 대 11.21%
- **예외:** Healthcare Representative는 초과근무 집단이 5.41%(2/37)로 비초과근무 집단 7.45%보다 낮다.
- **소표본(30명 미만):** Sales Representative·초과근무(24명, 66.67%), Human Resources·초과근무(13명), Manager·초과근무(27명), Research Director·초과근무(23명). 비율 차이를 해석할 때 주의한다.

## 8. 결과 해석

- **SQL 결과와 기존 분석:** 03번 EDA와 보고서 4장의 수치를 그대로 재현한다. 다음 절에서 pandas로 다시 계산해 확인한다.
- **이직률이 높은 집단:** 초과근무, 짧은 근속, 25세 미만, Sales Representative다.
- **교차 분석:** 같은 직무 안에서도 초과근무 여부에 따라 이직률이 크게 달라진다. 다만 소표본 칸이 있어 결론을 신중하게 내려야 한다.

## 9. 기존 pandas 분석과 비교

같은 원본 데이터로 pandas에서 같은 정의의 집계를 다시 계산해 SQL 결과와 비교한다.
- 연령대는 03·04번과 같은 `pd.cut(..., right=False)`를 쓴다.
- 근속 구간은 02번과 같은 `pd.cut`을 쓴다.

In [15]:
def pandas_rates(frame, column):
    grouped = frame.groupby(column, observed=True)["Attrition"]
    table = pd.DataFrame({
        "employee_count": grouped.size(),
        "attrition_count": grouped.apply(lambda s: int((s == "Yes").sum())),
    })
    table["attrition_rate"] = (100 * table["attrition_count"] / table["employee_count"]).round(2)
    return table


banded = hr.assign(
    AgeGroup=pd.cut(hr["Age"], bins=[0, 25, 35, 45, 55, float("inf")], right=False,
                    labels=["Under 25", "25-34", "35-44", "45-54", "55+"]).astype(str),
    TenureGroup=pd.cut(hr["YearsAtCompany"], bins=[-1, 1, 3, 5, 10, float("inf")],
                       labels=["0-1 Years", "2-3 Years", "4-5 Years", "6-10 Years", "Over 10 Years"]).astype(str),
)
pairs = {
    "부서": ("q3_department", "Department"),
    "성별": ("q4_gender", "Gender"),
    "직무": ("q5_job_role", "JobRole"),
    "초과근무": ("q6a_overtime", "OverTime"),
    "연령대": ("q7_age_group", "AgeGroup"),
    "근속 구간": ("q8a_tenure_years_at_company", "TenureGroup"),
}
rows = []
overall = results["q2_overall"].iloc[0]
total, left = len(hr), int((hr["Attrition"] == "Yes").sum())
rows.append(("전체", "전체", int(overall["employee_count"]), total, int(overall["attrition_count"]), left,
             float(overall["attrition_rate"]), round(100 * left / total, 2)))
for label, (query, column) in pairs.items():
    sql = results[query].set_index(column)
    pdf = pandas_rates(banded, column)
    for group, row in pdf.iterrows():
        rows.append((label, group, int(sql.loc[group, "employee_count"]), int(row["employee_count"]),
                     int(sql.loc[group, "attrition_count"]), int(row["attrition_count"]),
                     float(sql.loc[group, "attrition_rate"]), float(row["attrition_rate"])))
comparison = pd.DataFrame(rows, columns=["구분", "집단", "SQL 직원 수", "pandas 직원 수",
                                         "SQL 이직자 수", "pandas 이직자 수", "SQL 이직률", "pandas 이직률"])
comparison["일치"] = (
    (comparison["SQL 직원 수"] == comparison["pandas 직원 수"])
    & (comparison["SQL 이직자 수"] == comparison["pandas 이직자 수"])
    & (comparison["SQL 이직률"] == comparison["pandas 이직률"])
)
display(comparison)
print(f"비교 {len(comparison)}개 집단 중 일치 {int(comparison['일치'].sum())}개")
assert comparison["일치"].all(), "SQL과 pandas 결과가 다릅니다"

,구분,집단,SQL 직원 수,pandas 직원 수,SQL 이직자 수,pandas 이직자 수,SQL 이직률,pandas 이직률,일치
0,전체,전체,1470,1470,237,237,16.12,16.12,True
1,부서,Human Resources,63,63,12,12,19.05,19.05,True
2,부서,Research & Development,961,961,133,133,13.84,13.84,True
3,부서,Sales,446,446,92,92,20.63,20.63,True
4,성별,Female,588,588,87,87,14.80,14.80,True
5,성별,Male,882,882,150,150,17.01,17.01,True
6,직무,Healthcare Representative,131,131,9,9,6.87,6.87,True
7,직무,Human Resources,52,52,12,12,23.08,23.08,True
8,직무,Laboratory Technician,259,259,62,62,23.94,23.94,True
9,직무,Manager,102,102,5,5,4.90,4.90,True


비교 27개 집단 중 일치 27개


비교한 모든 집단(전체, 부서, 성별, 직무, 초과근무, 연령대, 근속 구간)에서 직원 수, 이직자 수, 이직률이 일치한다.
- 이 값들은 보고서 4.1~4.5의 표와도 같다.
- 근속 평균(Q8c)은 보고서 4.6의 평균과 같다.

## 10. 최종 결론

- **교차 검증:** SQL로 다시 집계한 결과가 pandas 분석, 보고서 수치와 모두 일치해 집단별 이직률 계산을 교차 검증했다.
- **추가로 확인한 점:**
  - 현 직무 근속이 짧은 집단(0~1년 27.91%)의 이직률이 높다.
  - 같은 직무 안에서도 초과근무 집단의 이직률이 대체로 높다.
- **한계:**
  - SQL 집계는 기술통계다. 다른 변수의 영향을 통제하지 않았고 유의성 검정도 하지 않았다.
  - 교차 분석의 30명 미만 칸은 비율이 크게 흔들릴 수 있다.
  - 데이터는 IBM이 만든 가상 데이터이며 기간 정보가 없어 연간 이직률로 해석할 수 없다.

In [16]:
con.close()
print("메모리 데이터베이스를 닫았습니다.")

메모리 데이터베이스를 닫았습니다.
